# Exact risk-set cohorts and timing probe in both transfer directions

Builds the time-matched and exact risk-set cohorts over the full window, reports the timing probe and the effect of adding the landmark time, and runs conformal analyses on the full-window risk-set cohort.

Data: PhysioNet/Computing in Cardiology Challenge 2019 training sets A and B (open access, CC BY 4.0), downloaded by the first code cell.

Run on Google Colab with a CPU runtime (Runtime > Run all). All results are written to `results_02.json`, which is downloaded at the end.

## Download the data

In [ ]:
import os, glob
!pip -q install awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/challenge-2019/1.0.0/training/ /content/data/
nA = len(glob.glob('/content/data/training_setA/*.psv')); nB = len(glob.glob('/content/data/training_setB/*.psv'))
print('Set A files:', nA, ' Set B files:', nB, ' (expected 20336 and 20000)')
if nA < 20336 or nB < 20000:
    print('S3 download incomplete - using wget instead (slower)')
    !wget -r -N -c -np -q https://physionet.org/files/challenge-2019/1.0.0/training/ -P /content/

## Load all ICU stays

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
DATA = None
for cand in ['/content/data', '/content/physionet.org/files/challenge-2019/1.0.0/training']:
    if cand and glob.glob(f'{cand}/training_setA/*.psv'):
        DATA = cand; break
assert DATA, 'PhysioNet files not found - run the download cell first'
print('Using data in', DATA)
DYN = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2','BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2',
       'AST','BUN','Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate',
       'Magnesium','Phosphate','Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
H = 6          # prediction horizon (hours)
W = 6          # feature window (hours)
W_MIN = 6      # minimum observed hours before a landmark
K_CTRL = 10    # controls per septic stay in risk-set sampling
SEEDS = [2026, 2027, 2028, 2029, 2030]
OUT = {}

def load_set(s):
    files = sorted(glob.glob(f'{DATA}/training_set{s}/*.psv'))
    stays = {}
    t0 = time.time()
    for f in files:
        d = pd.read_csv(f, sep='|')
        pid = os.path.basename(f)[:-4]
        lab = d['SepsisLabel'].to_numpy()
        pos = np.flatnonzero(lab == 1)
        stays[pid] = dict(X=d[DYN].to_numpy(np.float32), n=len(d),
                          first=(int(pos[0]) if len(pos) else -1),
                          age=float(d['Age'].iloc[0]), sex=float(d['Gender'].iloc[0]),
                          hadm=float(d['HospAdmTime'].iloc[0]), icu0=float(d['ICULOS'].iloc[0]),
                          unit=(0 if d['Unit1'].iloc[0] == 1 else (1 if d['Unit2'].iloc[0] == 1 else 2)))
    print(f'Set {s}: {len(stays)} stays loaded in {time.time()-t0:.0f}s')
    return stays

S = {s: load_set(s) for s in ['A', 'B']}
assert len(S['A']) > 0 and len(S['B']) > 0, 'Data not found'

## Feature and model helpers

In [ ]:
def features(st, lm):
    F = np.full((len(lm), 7 * len(DYN) + 4), np.nan, np.float32)
    meta = np.zeros((len(lm), 4), np.float32)   # age, sex, unit, tp (not model features)
    for r, (pid, y, tp) in enumerate(lm.itertuples(index=False)):
        v = st[pid]; hist = v['X'][:tp + 1]; win = hist[max(0, tp - W + 1):]
        with np.errstate(all='ignore'):
            ok = ~np.isnan(win); cnt = ok.sum(0)
            mean = np.nanmean(win, 0); sd = np.nanstd(win, 0); mn = np.nanmin(win, 0); mx = np.nanmax(win, 0)
            okh = ~np.isnan(hist)
            idx = np.where(okh.any(0), hist.shape[0] - 1 - np.argmax(okh[::-1], 0), -1)
            last = np.where(idx >= 0, hist[np.maximum(idx, 0), np.arange(hist.shape[1])], np.nan)
            t = np.arange(win.shape[0], dtype=np.float32)[:, None]
            tm = np.where(ok, t, 0).sum(0) / np.maximum(cnt, 1); vm = np.where(ok, win, 0).sum(0) / np.maximum(cnt, 1)
            cov = np.where(ok, (t - tm) * (np.nan_to_num(win) - vm), 0).sum(0)
            var = np.where(ok, (t - tm) ** 2, 0).sum(0)
            slope = np.where((cnt >= 2) & (var > 0), cov / np.where(var > 0, var, 1), np.nan)
            miss = 1 - cnt / win.shape[0]
        F[r, :7 * len(DYN)] = np.concatenate([mean, sd, mn, mx, last, slope, miss])
        F[r, -4:] = [v['age'], v['sex'], v['hadm'], miss.mean()]
        meta[r] = [v['age'], v['sex'], v['unit'], tp]
    return F, meta

def hgb(seed):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          max_depth=6, l2_regularization=1.0, class_weight='balanced',
                                          random_state=seed)

def wq_thresholds(cal_s, cal_w, test_w, alpha):
    """Weighted conformal quantile (Tibshirani et al. 2019) for each test point; inf if not attainable."""
    o = np.argsort(cal_s); s = cal_s[o]; cw = np.cumsum(cal_w[o]); tot = cw[-1]
    need = (1 - alpha) * (tot + test_w)
    k = np.searchsorted(cw, need, side='left')
    thr = np.full(len(test_w), np.inf); ok = k < len(s); thr[ok] = s[k[ok]]
    return thr

## Cohorts: time-matched and exact risk-set designs

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

def septic_cases(st, h):
    rows = []
    for pid, v in st.items():
        if v['first'] < 0: continue
        tp = v['first'] + 6 - h
        if W_MIN - 1 <= tp <= v['n'] - 1: rows.append((pid, 1, tp))
    return rows

def lm_trunc(st, h, seed):
    """Time-matched rule: each control landmark is drawn from septic landmarks that fall inside the stay."""
    rng = np.random.default_rng(seed); cases = septic_cases(st, h); ref = np.array([c[2] for c in cases])
    rows = list(cases)
    for pid, v in st.items():
        if v['first'] >= 0: continue
        cand = ref[ref <= v['n'] - 1]
        if len(cand): rows.append((pid, 0, int(rng.choice(cand))))
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp'])

def lm_riskset_exact(st, h, seed, k=K_CTRL):
    """Incidence-density sampling: for each septic stay with landmark t, k distinct non-septic stays still
    observed at t+h are sampled at the same t. A non-septic stay may serve in several risk sets (different t).
    Septic stays without k eligible controls are dropped, so every risk set has exactly k controls."""
    rng = np.random.default_rng(seed); cases = septic_cases(st, h)
    neg_pid = np.array([p for p, v in st.items() if v['first'] < 0])
    neg_n = np.array([st[p]['n'] for p in neg_pid])
    rows, dropped = [], 0
    for pid, _, t in cases:
        pool = np.flatnonzero(neg_n - 1 >= t + h)
        if len(pool) < k: dropped += 1; continue
        rows.append((pid, 1, t))
        rows += [(neg_pid[j], 0, t) for j in rng.choice(pool, size=k, replace=False)]
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), dropped

COHORT = {}; OUT['cohorts'] = {}
for s in ['A', 'B']:
    seed = 2019 + H + (0 if s == 'A' else 100)
    for name, lm in (('trunc', lm_trunc(S[s], H, seed)), ('riskset', None)):
        dropped = 0
        if name == 'riskset': lm, dropped = lm_riskset_exact(S[s], H, seed)
        X, meta = features(S[s], lm)
        COHORT[(name, s)] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), pid=lm.pid.to_numpy(), meta=meta)
        OUT['cohorts'][f'{name}_{s}'] = dict(rows=int(len(lm)), septic=int(lm.y.sum()), unique_stays=int(lm.pid.nunique()),
                                             septic_dropped_no_controls=int(dropped))
        print(f'{name:8s} {s}: rows {len(lm):6d}, septic {int(lm.y.sum()):5d}, unique stays {lm.pid.nunique():6d}, '
              f'septic dropped {dropped}')

## Leakage audit, both directions

In [ ]:
def boot_ci_auc_diff(y, p1, p2, pid, B=300, seed=0):
    rng = np.random.default_rng(seed); code = pd.factorize(pid)[0]; order = np.argsort(code, kind='stable')
    bounds = np.r_[0, np.cumsum(np.bincount(code))]; up = np.arange(len(bounds) - 1)
    idx = {u: order[bounds[u]:bounds[u + 1]] for u in up}; d = []
    for _ in range(B):
        i = np.concatenate([idx[u] for u in rng.choice(up, len(up))])
        if y[i].min() == y[i].max(): continue
        d.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]

g1 = {}
for name in ['trunc', 'riskset']:
    for src, tgt in (('A', 'B'), ('B', 'A')):
        A_, B_ = COHORT[(name, src)], COHORT[(name, tgt)]
        keep = (~np.isnan(A_['X'])).any(0) & (~np.isnan(B_['X'])).any(0)
        m = hgb(2026).fit(A_['X'][:, keep], A_['y']); p = m.predict_proba(B_['X'][:, keep])[:, 1]
        mt = hgb(2026).fit(np.column_stack([A_['X'][:, keep], A_['tp']]), A_['y'])
        pt = mt.predict_proba(np.column_stack([B_['X'][:, keep], B_['tp']]))[:, 1]
        r = dict(probe_tp_AUROC=float(roc_auc_score(B_['y'], B_['tp'])), prevalence=float(B_['y'].mean()),
                 AUROC=float(roc_auc_score(B_['y'], p)), AUPRC=float(average_precision_score(B_['y'], p)),
                 AUROC_with_tp=float(roc_auc_score(B_['y'], pt)), AUPRC_with_tp=float(average_precision_score(B_['y'], pt)),
                 delta_AUROC_tp_CI=boot_ci_auc_diff(B_['y'], pt, p, B_['pid']))
        g1[f'{name}_{src}to{tgt}'] = r
        print(f"{name:8s} {src}->{tgt}: tp-probe {r['probe_tp_AUROC']:.3f}  AUROC {r['AUROC']:.3f}  "
              f"+tp {r['AUROC_with_tp']:.3f}  AUPRC {r['AUPRC']:.3f}  prev {r['prevalence']:.3f}")
OUT['timing_audit_exact'] = g1

## Conformal: source-only vs restricted weighting vs target label budget

In [ ]:
VIT = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
NC = len(DYN)
RESTRICT = [b * NC + DYN.index(c) for b in (0, 2, 3, 4) for c in VIT] + [7 * NC]   # mean,min,max,last of vitals + age

def split_by_stay(y, pid, frac, seed):
    rng = np.random.default_rng(seed); cal = np.zeros(len(y), bool)
    for lab in (0, 1):
        u = np.unique(pid[y == lab]); pick = set(rng.choice(u, int(round(frac * len(u))), replace=False))
        cal |= np.isin(pid, list(pick)) & (y == lab)
    return ~cal, cal

def mondrian_thr(s1, s0, a1, a0):
    out = []
    for c, a in ((s1, a1), (s0, a0)):
        c = np.sort(c); k = int(np.ceil((len(c) + 1) * (1 - a))) - 1
        out.append(c[k] if k < len(c) else np.inf)
    return out

def summarize(inc1, inc0, y):
    alert = inc1 & ~inc0; defer = inc1 & inc0
    return dict(septic_cov=float(inc1[y == 1].mean()), nonseptic_cov=float(inc0[y == 0].mean()),
                defer_rate=float(defer.mean()), alert_rate=float(alert.mean()),
                empty_rate=float((~inc1 & ~inc0).mean()),
                alert_PPV=(float(y[alert].mean()) if alert.any() else float('nan')))

def domain_weights(Xs, Xt, Xeval):
    allX = np.vstack([Xs, Xt]); med = np.nan_to_num(np.nanmedian(allX, 0), nan=0.0)
    f = lambda Z: np.nan_to_num(np.where(np.isnan(Z), med, Z), nan=0.0, posinf=0.0, neginf=0.0)
    sc = StandardScaler().fit(f(allX)); d = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))]
    lr = LogisticRegression(C=0.1, max_iter=3000).fit(sc.transform(f(allX)), d)
    pe = np.clip(lr.predict_proba(sc.transform(f(Xeval)))[:, 1], 1e-6, 1 - 1e-6)
    return pe / (1 - pe) * len(Xs) / len(Xt), float(roc_auc_score(d, lr.predict_proba(sc.transform(f(allX)))[:, 1]))

A0S = (0.10, 0.20, 0.30); A1 = 0.10
BUDGETS = [10, 20, 30, 50, 100]; DRAWS = 30
g2 = {}
for name in ['trunc', 'riskset']:
    for src, tgt in (('A', 'B'), ('B', 'A')):
        C_s, C_t = COHORT[(name, src)], COHORT[(name, tgt)]
        keep = (~np.isnan(C_s['X'])).any(0) & (~np.isnan(C_t['X'])).any(0)
        res = {}
        for seed in SEEDS:
            tr, cc = split_by_stay(C_s['y'], C_s['pid'], 0.3, seed)
            m = hgb(seed).fit(C_s['X'][tr][:, keep], C_s['y'][tr])
            pc = m.predict_proba(C_s['X'][cc][:, keep])[:, 1]; yc = C_s['y'][cc]
            pt = m.predict_proba(C_t['X'][:, keep])[:, 1]; yt = C_t['y']
            r = dict(AUROC=float(roc_auc_score(yt, pt)))
            # (a) source-only Mondrian (the guarantee is for the source hospital)
            for a0 in A0S:
                t1, t0 = mondrian_thr(1 - pc[yc == 1], pc[yc == 0], A1, a0)
                r[f'source_mondrian_a0_{a0}'] = summarize(1 - pt <= t1, pt <= t0, yt)
            if name == 'riskset':
                # (b) importance weighting on a restricted, commonly measured feature set
                half = np.random.default_rng(seed).random(len(yt)) < 0.5
                wt = np.zeros(len(yt)); wc = np.zeros(len(yc)); dauc = []
                for h_ in (half, ~half):
                    w_e, a_ = domain_weights(C_s['X'][tr][:, RESTRICT], C_t['X'][~h_][:, RESTRICT],
                                             np.vstack([C_t['X'][h_][:, RESTRICT], C_s['X'][cc][:, RESTRICT]]))
                    wt[h_] = w_e[:h_.sum()]; wc += 0.5 * w_e[h_.sum():]; dauc.append(a_)
                cap = np.quantile(np.r_[wc, wt], 0.99); wc = np.minimum(wc, cap); wt = np.minimum(wt, cap)
                r['restricted_domain_AUROC'] = float(np.mean(dauc))
                r['restricted_ESS_septic'] = float(wc[yc == 1].sum() ** 2 / (wc[yc == 1] ** 2).sum() / (yc == 1).sum())
                t1w = wq_thresholds(1 - pc[yc == 1], wc[yc == 1], wt, A1); t0w = wq_thresholds(pc[yc == 0], wc[yc == 0], wt, 0.10)
                r['restricted_weighted_mondrian_a0_0.1'] = summarize(1 - pt <= t1w, pt <= t0w, yt)
                # (c) target label budget: n septic + 10n non-septic labelled target stays recalibrate Mondrian
                rng = np.random.default_rng(seed + 7)
                tid = pd.factorize(C_t['pid'])[0]
                pos_p = np.unique(tid[yt == 1]); neg_p = np.unique(tid[yt == 0])
                for nb in [b for b in BUDGETS if b <= len(pos_p) // 2 and 10 * b <= len(neg_p) // 2]:
                    for a0 in A0S:
                        draws = []
                        for _ in range(DRAWS):
                            cp = rng.choice(pos_p, nb, replace=False); cn = rng.choice(neg_p, min(10 * nb, len(neg_p)), replace=False)
                            calm = (np.isin(tid, cp) & (yt == 1)) | (np.isin(tid, cn) & (yt == 0))
                            evm = ~np.isin(tid, np.r_[cp, cn])
                            t1, t0 = mondrian_thr(1 - pt[calm & (yt == 1)], pt[calm & (yt == 0)], A1, a0)
                            draws.append(summarize(1 - pt[evm] <= t1, pt[evm] <= t0, yt[evm]))
                        cov = np.array([d['septic_cov'] for d in draws])
                        r[f'target_budget_{nb}_a0_{a0}'] = dict(
                            {k: float(np.nanmean([d[k] for d in draws])) for k in draws[0]},
                            septic_cov_p10=float(np.percentile(cov, 10)), frac_draws_cov_ge_085=float(np.mean(cov >= 0.85)))
            res[seed] = r
        keys = [k for k in res[SEEDS[0]] if all(k in res[s_] for s_ in SEEDS)]
        agg = {}
        for k in keys:
            v0 = res[SEEDS[0]][k]
            if isinstance(v0, dict):
                agg[k] = {m: [float(np.nanmean([res[s][k][m] for s in SEEDS])), float(np.nanmin([res[s][k][m] for s in SEEDS])),
                              float(np.nanmax([res[s][k][m] for s in SEEDS]))] for m in v0}
            else:
                agg[k] = [float(np.mean([res[s][k] for s in SEEDS])), float(np.min([res[s][k] for s in SEEDS])), float(np.max([res[s][k] for s in SEEDS]))]
        g2[f'{name}_{src}to{tgt}'] = agg
        print(f"\n{name} {src}->{tgt}  AUROC {agg['AUROC'][0]:.3f}")
        for k in keys:
            if isinstance(agg[k], dict) and ('a0_0.1' in k):
                v = agg[k]
                print(f"  {k:40s} septic cov {v['septic_cov'][0]:.3f} [{v['septic_cov'][1]:.2f}-{v['septic_cov'][2]:.2f}]"
                      f"  defer {v['defer_rate'][0]:.3f}  alert {v['alert_rate'][0]:.3f}")
OUT['conformal_full_window'] = g2

## Save results

In [ ]:
with open('results_02.json', 'w') as fh:
    json.dump(OUT, fh, indent=1)
print('Saved results_02.json')

In [ ]:
from google.colab import files
files.download('results_02.json')